# Pipeline completo: treino (recapitulado) + segmentacao + classificacao

Trabalho de Inteligencia Artificial -- reaproveitando o pipeline do TCC.

Esse notebook tem duas partes:

1. **Treinamento (recapitulado)** dos dois modelos -- segmentacao de instancia
   (milho/null) e classificacao (healthy/unhealthy). O codigo de treino esta aqui
   de verdade (mesmos hiperparametros usados no TCC), mas como os dois modelos ja
   foram treinados antes e o backup esta salvo no Drive, as celulas carregam os
   pesos prontos em vez de treinar de novo -- assim o notebook roda rapido. Se
   algum dia precisar retreinar do zero (ex: trocou o dataset), e' so' apagar o
   backup do Drive que o treino completo dispara sozinho.
2. **Pipeline de inferencia** numa pasta especifica de fotos escolhidas a dedo
   (nao o split de teste inteiro) -- pra cada foto: acha as plantas de milho
   (segmentacao), isola cada uma (mascara + recorte) e classifica cada planta
   como saudavel ou nao. Regra de agregacao do TCC: uma planta `unhealthy` ja
   marca a foto inteira como `unhealthy`.

Antes de rodar: **Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU (T4)**.

In [ ]:
!pip install -q ultralytics pycocotools
import ultralytics
ultralytics.checks()

## Monta o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Confere se a GPU esta ativa

Se der erro aqui, vai em **Ambiente de execucao > Alterar tipo de ambiente de
execucao > GPU (T4)**, reconecta e roda de novo.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "GPU nao detectada. Ve em 'Ambiente de execucao > Alterar tipo de ambiente de execucao' "
        "e escolhe GPU (T4), depois reconecta o runtime e roda essa celula de novo."
    )
print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

---
## 1. Treinamento -- Segmentacao milho/null

Instance segmentation (`yolo26n-seg.pt`, fine-tuning), `imgsz=640`, `batch=16`,
`epochs=100` com `patience=20`. Dataset exportado do Roboflow em formato COCO
Segmentation.

Essa celula primeiro tenta restaurar o backup do Drive (`relatorio_treino_segmentacao_completo.zip`).
Se achar `best.pt` ai dentro, carrega direto e **nao treina de novo**. Se nao achar,
ai sim baixa o dataset bruto, converte COCO -> YOLO-seg e treina do zero (isso demora
bem mais, ~1h30-2h).

In [ ]:
import os
from ultralytics import YOLO

SEG_NAME = "milho_corn_seg"
SEG_RUN_DIR = f"runs/segment/{SEG_NAME}"
SEG_BACKUP_ZIP = "/content/drive/MyDrive/milho/relatorio_treino_segmentacao_completo.zip"
SEG_BEST_PT = f"{SEG_RUN_DIR}/weights/best.pt"

os.makedirs(SEG_RUN_DIR, exist_ok=True)
if not os.path.exists(SEG_BEST_PT) and os.path.exists(SEG_BACKUP_ZIP):
    print("Restaurando backup de segmentacao do Drive...")
    !unzip -q -o "{SEG_BACKUP_ZIP}" -d {SEG_RUN_DIR}

if os.path.exists(SEG_BEST_PT):
    print(f"Pesos de segmentacao ja existem em {SEG_BEST_PT}, carregando SEM retreinar.")
    model_seg = YOLO(SEG_BEST_PT)
else:
    print("Backup nao encontrado -- preparando dataset e treinando do zero (vai demorar).")

    # --- descompacta o dataset COCO bruto ---
    SEG_ZIP_PATH = "/content/drive/MyDrive/milho/blick-corn-analyzer.zip"
    RAW_DIR = "/content/coco_raw"
    os.makedirs(RAW_DIR, exist_ok=True)
    !unzip -q "{SEG_ZIP_PATH}" -d {RAW_DIR}

    # --- localiza o json de cada split ---
    import glob, json as json_lib
    def encontra_json(padrao):
        return [j for j in glob.glob(f"{RAW_DIR}/**/*.json", recursive=True)
                if padrao in os.path.basename(os.path.dirname(j)).lower()]

    splits_json = {}
    for nome, padrao in [("train", "train"), ("valid", "val"), ("test", "test")]:
        encontrados = encontra_json(padrao)
        assert len(encontrados) == 1, f"Esperava 1 json pra '{nome}', achei {len(encontrados)}: {encontrados}"
        splits_json[nome] = encontrados[0]
    print("Splits encontrados:", splits_json)

    # --- converte COCO -> YOLO-seg ---
    from pathlib import Path
    import shutil, cv2
    import numpy as np
    from pycocotools import mask as mask_utils

    SEG_DATASET_DIR = Path("/content/dataset_seg")

    def segmentacao_para_poligonos(seg, img_w, img_h):
        poligonos = []
        if isinstance(seg, dict):
            try:
                rle = mask_utils.frPyObjects(seg, img_h, img_w) if isinstance(seg.get("counts"), list) else seg
                mascara = mask_utils.decode(rle)
                if mascara.ndim == 3:
                    mascara = mascara[:, :, 0]
                contornos, _ = cv2.findContours(mascara.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for c in contornos:
                    if len(c) >= 3:
                        poligonos.append(c.reshape(-1, 2).astype(float))
            except Exception as e:
                print(f"Aviso: falha ao decodificar RLE ({e})")
        elif isinstance(seg, list):
            for parte in seg:
                if isinstance(parte, list) and len(parte) >= 6:
                    poligonos.append(np.array(parte, dtype=float).reshape(-1, 2))
        return poligonos

    def coco_to_yolo_seg(json_path, imagens_dir, out_images_dir, out_labels_dir, categoria_para_idx):
        with open(json_path) as f:
            d = json_lib.load(f)
        out_images_dir.mkdir(parents=True, exist_ok=True)
        out_labels_dir.mkdir(parents=True, exist_ok=True)
        imagens_por_id = {img["id"]: img for img in d["images"]}
        anotacoes_por_imagem = {}
        for ann in d["annotations"]:
            anotacoes_por_imagem.setdefault(ann["image_id"], []).append(ann)

        for img_id, img in imagens_por_id.items():
            origem = imagens_dir / img["file_name"]
            if not origem.exists():
                continue
            shutil.copy(origem, out_images_dir / img["file_name"])

            anns = anotacoes_por_imagem.get(img_id, [])
            if not anns:
                continue
            linhas = []
            w, h = img["width"], img["height"]
            for ann in anns:
                classe_idx = categoria_para_idx[ann["category_id"]]
                seg = ann.get("segmentation")
                if not seg:
                    continue
                for pts in segmentacao_para_poligonos(seg, w, h):
                    coords = [f"{v:.6f}" for x, y in pts for v in (x / w, y / h)]
                    linhas.append(str(classe_idx) + " " + " ".join(coords))
            if linhas:
                (out_labels_dir / (Path(img["file_name"]).stem + ".txt")).write_text("\n".join(linhas))

    with open(splits_json["train"]) as f:
        categorias_ordenadas = sorted(json_lib.load(f)["categories"], key=lambda c: c["id"])
    categoria_para_idx = {c["id"]: i for i, c in enumerate(categorias_ordenadas)}
    nomes_classes_seg = [c["name"] for c in categorias_ordenadas]
    print("Classes:", nomes_classes_seg)

    for nome, json_path in splits_json.items():
        coco_to_yolo_seg(
            json_path=json_path,
            imagens_dir=Path(json_path).parent,
            out_images_dir=SEG_DATASET_DIR / "images" / nome,
            out_labels_dir=SEG_DATASET_DIR / "labels" / nome,
            categoria_para_idx=categoria_para_idx,
        )

    data_yaml = f"path: {SEG_DATASET_DIR}\ntrain: images/train\nval: images/valid\ntest: images/test\n\nnames:\n"
    for i, nome_classe in enumerate(nomes_classes_seg):
        data_yaml += f"  {i}: {nome_classe}\n"
    SEG_DATA_YAML = SEG_DATASET_DIR / "data.yaml"
    SEG_DATA_YAML.write_text(data_yaml)

    # --- treina ---
    model_seg = YOLO("yolo26n-seg.pt")
    model_seg.train(
        data=str(SEG_DATA_YAML),
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
        name=SEG_NAME,
        device=0,
    )

    import shutil as shutil_lib
    shutil_lib.make_archive("/content/drive/MyDrive/milho/relatorio_treino_segmentacao_completo", "zip", SEG_RUN_DIR)
    print("Backup de segmentacao salvo no Drive.")

print("Classes do modelo de segmentacao:", model_seg.names)

IDX_CLASSE_CORN = None
for idx, nome in model_seg.names.items():
    if "corn" in nome.lower() or "milho" in nome.lower():
        IDX_CLASSE_CORN = idx
        break
assert IDX_CLASSE_CORN is not None, "Nao achei uma classe 'corn'/'milho' -- confere model_seg.names e ajusta IDX_CLASSE_CORN na mao."
print("Indice da classe corn:", IDX_CLASSE_CORN)

---
## 2. Treinamento -- Classificacao healthy/unhealthy

Classificacao (`yolo26n-cls.pt`, fine-tuning), `imgsz=224`, `batch=64`, `epochs=100`
com `patience=20`, `cache="ram"`. Dataset exportado do Roboflow em formato Folder
Structure (pastas `healthy/` e `unhealthy/` dentro de cada split).

Mesma logica: restaura o backup do Drive (`relatorio_treino_completo.zip`) e, se
achar `best.pt`, carrega sem retreinar. So' baixa o dataset e treina do zero se o
backup nao existir.

In [ ]:
CLS_NAME = "milho_healthy_unhealthy"
CLS_RUN_DIR = f"runs/classify/{CLS_NAME}"
CLS_BACKUP_ZIP = "/content/drive/MyDrive/milho/relatorio_treino_completo.zip"
CLS_BEST_PT = f"{CLS_RUN_DIR}/weights/best.pt"

os.makedirs(CLS_RUN_DIR, exist_ok=True)
if not os.path.exists(CLS_BEST_PT) and os.path.exists(CLS_BACKUP_ZIP):
    print("Restaurando backup de classificacao do Drive...")
    !unzip -q -o "{CLS_BACKUP_ZIP}" -d {CLS_RUN_DIR}

if os.path.exists(CLS_BEST_PT):
    print(f"Pesos de classificacao ja existem em {CLS_BEST_PT}, carregando SEM retreinar.")
    model_cls = YOLO(CLS_BEST_PT)
else:
    print("Backup nao encontrado -- preparando dataset e treinando do zero (vai demorar).")

    CLS_ZIP_PATH = "/content/drive/MyDrive/milho/blick-healthy-unhealthy.folder.zip"
    CLS_DATASET_DIR = "/content/dataset_cls"
    os.makedirs(CLS_DATASET_DIR, exist_ok=True)
    !unzip -q "{CLS_ZIP_PATH}" -d {CLS_DATASET_DIR}

    model_cls = YOLO("yolo26n-cls.pt")
    model_cls.train(
        data=CLS_DATASET_DIR,
        epochs=100,
        imgsz=224,
        batch=64,
        patience=20,
        name=CLS_NAME,
        device=0,
        cache="ram",
    )

    import shutil as shutil_lib
    shutil_lib.make_archive("/content/drive/MyDrive/milho/relatorio_treino_completo", "zip", CLS_RUN_DIR)
    print("Backup de classificacao salvo no Drive.")

print("Classes do modelo de classificacao:", model_cls.names)

---
## 3. Pipeline de inferencia -- so' nas fotos escolhidas

Diferente da ultima versao, aqui NAO usa o split de teste inteiro -- so' roda nas
fotos que voce escolheu a dedo (fica mais facil de olhar resultado por resultado e
decidir se o pipeline ta acertando).

A pasta `blick-fotos-09-09` esta no seu PC (`C:\Users\Mauro\Downloads\blick-fotos-09-09`),
e o Colab so' enxerga arquivos que estao no Drive ou que voce sobe manualmente. Duas
opcoes, usa uma das duas:

**Opcao A -- Drive (recomendado, mais estavel):** copia a pasta inteira
`blick-fotos-09-09` pra dentro de `MyDrive/milho/` no seu Google Drive (arrasta e
solta no drive.google.com). Depois roda a celula de baixo normalmente.

**Opcao B -- upload direto nessa sessao do Colab:** descomenta e roda o bloco
`files.upload()` na celula de baixo -- abre uma janela pra voce selecionar os
arquivos da pasta no seu PC direto (precisa fazer de novo toda vez que a sessao do
Colab reiniciar, por isso a opcao A e' melhor pra usar varias vezes).

In [ ]:
from pathlib import Path

EXTENSOES_IMAGEM = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# --- Opcao A: pasta no Drive (padrao) ---
IMAGENS_DIR = Path("/content/drive/MyDrive/milho/blick-fotos-09-09")

# --- Opcao B: upload manual nessa sessao -- descomenta se preferir essa opcao ---
# from google.colab import files
# IMAGENS_DIR = Path("/content/blick-fotos-09-09")
# IMAGENS_DIR.mkdir(parents=True, exist_ok=True)
# enviados = files.upload()
# for nome_arquivo, conteudo in enviados.items():
#     (IMAGENS_DIR / nome_arquivo).write_bytes(conteudo)

assert IMAGENS_DIR.exists(), (
    f"Nao achei a pasta {IMAGENS_DIR}. Copia a pasta 'blick-fotos-09-09' do seu PC "
    f"pra dentro de MyDrive/milho/ no Google Drive (Opcao A), ou usa a Opcao B acima."
)

imagens_selecionadas = sorted([p for p in IMAGENS_DIR.iterdir() if p.suffix.lower() in EXTENSOES_IMAGEM])
assert imagens_selecionadas, f"A pasta {IMAGENS_DIR} existe mas nao tem nenhuma imagem dentro."
print(f"Imagens encontradas ({len(imagens_selecionadas)}):")
for p in imagens_selecionadas:
    print(" ", p.name)

## Isola cada planta e classifica

Mesmo processo de sempre: segmenta a foto inteira (ignora instancias `null`), isola
cada planta encontrada (mascara + recorte no bounding box com margem, igual o
`isolate_corn_plants.py`) e classifica cada uma como `healthy`/`unhealthy`.

**Veredito da imagem**: uma unica planta `unhealthy` ja marca a foto inteira como
`unhealthy` (regra do TCC). So' fica `healthy` se todas as plantas detectadas
estiverem saudaveis. Se nao detectar nenhuma planta, o veredito fica
`sem milho detectado`.

In [ ]:
import cv2
import numpy as np


def isolar_instancias(imagem, poligonos, padding=15):
    altura, largura = imagem.shape[:2]
    saida = []
    for poligono in poligonos:
        pontos = np.round(poligono).astype(np.int32)
        if len(pontos) < 3:
            continue

        mascara = np.zeros((altura, largura), dtype=np.uint8)
        cv2.fillPoly(mascara, [pontos], 255)
        isolado = cv2.bitwise_and(imagem, imagem, mask=mascara)

        x, y, w, h = cv2.boundingRect(mascara)
        x1, y1 = max(0, x - padding), max(0, y - padding)
        x2, y2 = min(largura, x + w + padding), min(altura, y + h + padding)
        crop = isolado[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        saida.append({"crop": crop, "bbox": (x1, y1, x2, y2), "poligono": pontos})
    return saida


def processar_imagem(caminho_imagem, conf_seg=0.4, padding=15):
    imagem = cv2.imread(str(caminho_imagem))
    if imagem is None:
        raise IOError(f"Nao consegui abrir {caminho_imagem}")

    resultado_seg = model_seg.predict(imagem, conf=conf_seg, verbose=False)[0]

    poligonos_corn = []
    if resultado_seg.masks is not None:
        classes_det = resultado_seg.boxes.cls.cpu().numpy().astype(int)
        confs_det = resultado_seg.boxes.conf.cpu().numpy()
        for poligono, classe_idx, conf_det in zip(resultado_seg.masks.xy, classes_det, confs_det):
            if classe_idx == IDX_CLASSE_CORN:
                poligonos_corn.append((poligono, float(conf_det)))

    instancias = []
    if poligonos_corn:
        isolados = isolar_instancias(imagem, [p for p, _ in poligonos_corn], padding=padding)
        for isolado, (_, conf_det) in zip(isolados, poligonos_corn):
            resultado_cls = model_cls.predict(isolado["crop"], verbose=False)[0]
            instancias.append({
                "poligono": isolado["poligono"],
                "bbox": isolado["bbox"],
                "conf_deteccao": conf_det,
                "classe_saude": resultado_cls.names[resultado_cls.probs.top1],
                "conf_saude": float(resultado_cls.probs.top1conf),
            })

    if not instancias:
        veredito = "sem milho detectado"
    elif any(inst["classe_saude"] == "unhealthy" for inst in instancias):
        veredito = "unhealthy"
    else:
        veredito = "healthy"

    return {"caminho": str(caminho_imagem), "imagem": imagem, "instancias": instancias, "veredito": veredito}

## Roda o pipeline nas fotos selecionadas

In [ ]:
resultados = [processar_imagem(caminho) for caminho in imagens_selecionadas]

for r in resultados:
    n_healthy = sum(1 for i in r["instancias"] if i["classe_saude"] == "healthy")
    n_unhealthy = sum(1 for i in r["instancias"] if i["classe_saude"] == "unhealthy")
    print(f"{Path(r['caminho']).name:45s} -> {len(r['instancias'])} planta(s) "
          f"({n_healthy} healthy, {n_unhealthy} unhealthy) -- veredito: {r['veredito'].upper()}")

## Visualizacao: contorno de cada planta + veredito da imagem

Verde = `healthy`, vermelho = `unhealthy`. O titulo de cada imagem mostra o
veredito final (regra de agregacao do TCC).

In [ ]:
import matplotlib.pyplot as plt

COR_HEALTHY = (0, 200, 0)
COR_UNHEALTHY = (220, 0, 0)


def desenhar_resultado(resultado):
    imagem_rgb = cv2.cvtColor(resultado["imagem"], cv2.COLOR_BGR2RGB).copy()
    for inst in resultado["instancias"]:
        cor = COR_HEALTHY if inst["classe_saude"] == "healthy" else COR_UNHEALTHY
        cv2.polylines(imagem_rgb, [inst["poligono"]], isClosed=True, color=cor, thickness=3)
        x1, y1, _, _ = inst["bbox"]
        texto = f"{inst['classe_saude']} {inst['conf_saude']:.0%}"
        cv2.putText(imagem_rgb, texto, (x1, max(0, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor, 2, cv2.LINE_AA)
    return imagem_rgb


n_col = min(4, len(resultados))
n_lin = (len(resultados) + n_col - 1) // n_col
fig, axes = plt.subplots(n_lin, n_col, figsize=(5 * n_col, 5 * n_lin))
axes = np.array(axes).reshape(-1)

for ax, resultado in zip(axes, resultados):
    ax.imshow(desenhar_resultado(resultado))
    cor_titulo = "red" if resultado["veredito"] == "unhealthy" else ("green" if resultado["veredito"] == "healthy" else "gray")
    ax.set_title(f"{Path(resultado['caminho']).name}\nveredito: {resultado['veredito'].upper()}", color=cor_titulo, fontsize=10)
    ax.axis("off")

for ax in axes[len(resultados):]:
    ax.axis("off")

plt.suptitle("Pipeline: segmentacao + isolamento + classificacao por planta", fontsize=13)
plt.tight_layout()

OUT_DIR = "pipeline_completo_resultados"
os.makedirs(OUT_DIR, exist_ok=True)
saida = f"{OUT_DIR}/grid_resultados.png"
plt.savefig(saida, dpi=150)
print(f"Salvo em: {saida}")
plt.show()

## Resumo tabular + imagens individuais anotadas + backup

CSV com uma linha por planta detectada, imagens anotadas separadas, e zip salvo no
Drive + download direto.

In [ ]:
import csv as csv_module
import shutil
from google.colab import files

linhas = []
for resultado in resultados:
    nome_imagem = Path(resultado["caminho"]).name
    imagem_anotada = desenhar_resultado(resultado)
    cv2.imwrite(f"{OUT_DIR}/{Path(nome_imagem).stem}_anotada.png", cv2.cvtColor(imagem_anotada, cv2.COLOR_RGB2BGR))

    if not resultado["instancias"]:
        linhas.append({"imagem": nome_imagem, "planta_num": "", "conf_deteccao": "",
                        "classe_saude": "", "conf_saude": "", "veredito_imagem": resultado["veredito"]})
    for i, inst in enumerate(resultado["instancias"], 1):
        linhas.append({
            "imagem": nome_imagem, "planta_num": i,
            "conf_deteccao": f"{inst['conf_deteccao']:.4f}",
            "classe_saude": inst["classe_saude"],
            "conf_saude": f"{inst['conf_saude']:.4f}",
            "veredito_imagem": resultado["veredito"],
        })

csv_path = f"{OUT_DIR}/resumo_pipeline.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    campos = ["imagem", "planta_num", "conf_deteccao", "classe_saude", "conf_saude", "veredito_imagem"]
    w = csv_module.DictWriter(f, fieldnames=campos)
    w.writeheader()
    w.writerows(linhas)
print(f"CSV salvo em: {csv_path}")

n_unhealthy_final = sum(1 for r in resultados if r["veredito"] == "unhealthy")
n_healthy_final = sum(1 for r in resultados if r["veredito"] == "healthy")
n_sem_deteccao = sum(1 for r in resultados if r["veredito"] == "sem milho detectado")
print(f"Resumo final: {len(resultados)} imagens -- {n_healthy_final} healthy, "
      f"{n_unhealthy_final} unhealthy, {n_sem_deteccao} sem milho detectado")

destino_base = "/content/drive/MyDrive/milho/pipeline_completo_resultados"
os.makedirs(os.path.dirname(destino_base), exist_ok=True)
zip_path = shutil.make_archive(destino_base, "zip", OUT_DIR)
print("Salvo no Drive:", zip_path)
files.download(zip_path)